# 01. Dataset sintético de riesgo de facturas

Este notebook genera y revisa `invoice-risk-v1`. Los archivos se crean localmente en `data/`, un directorio ignorado por Git.

## Objetivo y variables

El target es `manual_review_required`: indica si una factura sintética requiere revisión manual. Las ocho `MODEL_FEATURES` son:

- `invoice_amount_cents`
- `vendor_tenure_days`
- `previous_incidents_12m`
- `amount_vs_vendor_median`
- `has_purchase_order`
- `three_way_match`
- `bank_account_recently_changed`
- `country_risk`

`invoice_id` y `submitted_at` se conservan para trazabilidad; no son features. `submitted_at` también define el orden cronológico de las particiones.

In [ ]:
import csv
import json
from collections import Counter

from invoiceops_ml.data import (
    DATASET_VERSION,
    MODEL_FEATURES,
    SPLIT_FILENAMES,
    TARGET,
    generate_synthetic_dataset,
)

SEED = 202604
ROWS = 12_000
dataset_dir = generate_synthetic_dataset(seed=SEED, rows=ROWS)
dataset_dir

## Metadata y esquema

La seed explícita permite reproducir exactamente los CSV y `metadata.json`. Los metadatos registran la versión, el esquema, el target, los tamaños y los hashes SHA-256 de cada split.

In [ ]:
metadata = json.loads((dataset_dir / 'metadata.json').read_text(encoding='utf-8'))

assert metadata['dataset_version'] == DATASET_VERSION
assert metadata['seed'] == SEED
assert metadata['target'] == TARGET
print(json.dumps(metadata, indent=2, sort_keys=True))
print('Features:', ', '.join(MODEL_FEATURES))

## Partición cronológica

Los registros se ordenan por `submitted_at` antes de dividirlos en 70% train, 15% validation y 15% test. No se mezclan filas futuras en splits anteriores.

Evite leakage: no use campos que revelen una decisión posterior a `submitted_at`, ni reutilice `invoice_id` como feature. Las transformaciones que dependan de datos históricos deben respetar el límite temporal de cada split.

In [ ]:
def read_split(filename: str) -> list[dict[str, str]]:
    with (dataset_dir / filename).open(newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))

splits = {filename.removesuffix('.csv'): read_split(filename) for filename in SPLIT_FILENAMES}
train, validation, test = (splits[name] for name in ('train', 'validation', 'test'))

assert train[-1]['submitted_at'] < validation[0]['submitted_at'] < test[0]['submitted_at']
assert all(
    set(rows[0]) == {'invoice_id', 'submitted_at', *MODEL_FEATURES, TARGET}
    for rows in splits.values()
)
for name, rows in splits.items():
    print(f'{name}: {len(rows)} filas, desde {rows[0]["submitted_at"]} hasta {rows[-1]["submitted_at"]}')

## Distribución del target

Revise la proporción de `manual_review_required` en cada split antes de continuar con las siguientes actividades.

In [ ]:
for name, rows in splits.items():
    counts = Counter(row[TARGET] for row in rows)
    total = len(rows)
    distribution = {label: f'{count / total:.1%}' for label, count in sorted(counts.items())}
    print(f'{name}: {distribution}')